# 03.02 — Cypher Generation (DDL & DML)

Orthograph can generate Cypher queries from your model definitions, similar to how an ORM generates SQL. This ensures that your queries are consistent with your model and reduces boilerplate when working with Neo4j or other Cypher-compatible databases.

This notebook covers:
- Creating a `CypherGenerator` from a `GraphDefinition`
- Generating MERGE and CREATE queries for nodes
- Generating relationship queries
- Generating uniqueness constraints
- Generating MATCH patterns
- A full workflow combining all operations
- Validating generated queries against the model
- Emitting typed query objects for use in a catalogue

In [1]:
from typing import Optional

from shared.filmography import ActedIn, City, Directed, LivesIn, Movie, Person

from orthograph.cypher.generator import CypherGenerator
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import NodeModel, RelationshipModel

## Creating the generator

The `CypherGenerator` takes a `GraphDefinition` and provides methods for generating Cypher queries.

In [2]:
graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie, City],
    relationship_types=[ActedIn, Directed, LivesIn],
)

In [3]:
gen = CypherGenerator(graph_definition)
print("CypherGenerator ready for model:", graph_definition.name)

CypherGenerator ready for model: Filmography


## Generating MERGE queries

MERGE uses the `uid_field` to match an existing node. If the node exists, it updates the remaining properties via SET. If it does not exist, it creates it. This is the idempotent way to upsert nodes.

The data dict must include `__label__` and any properties to set.

In [4]:
# MERGE a Person node
query, params = gen.merge_node(
    {
        "__label__": "Person",
        "name": "Keanu Reeves",
        "born": 1964,
    }
)
print("Query: ", query)
print("Params:", params)
print()

# MERGE a Movie node
query, params = gen.merge_node(
    {
        "__label__": "Movie",
        "title": "The Matrix",
        "year": 1999,
    }
)
print("Query: ", query)
print("Params:", params)

Query:  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n
Params: {'name': 'Keanu Reeves', 'born': 1964}

Query:  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n
Params: {'title': 'The Matrix', 'year': 1999}


## Generating CREATE queries

Use `create_node` for unconditional creation -- when you know the node does not yet exist, or when you do not need idempotency.

In [5]:
query, params = gen.create_node(
    {
        "__label__": "City",
        "name": "Los Angeles",
    }
)
print("Query: ", query)
print("Params:", params)

Query:  CREATE (n:City {name: $name}) RETURN n
Params: {'name': 'Los Angeles'}


## Relationship queries

Both `create_relationship` and `merge_relationship` generate queries that first MATCH the source and target nodes by their UID fields, then CREATE or MERGE the relationship between them.

The data dict must include:
- `__label__` -- the relationship label
- `__source_uid__` -- the UID value of the source node
- `__target_uid__` -- the UID value of the target node
- Any additional keys are treated as relationship properties

In [6]:
# CREATE a relationship with properties (ACTED_IN has a 'role' property)
query, params = gen.create_relationship(
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    }
)
print("CREATE relationship:")
print("  Query: ", query)
print("  Params:", params)
print()

# MERGE a relationship without properties (DIRECTED has no properties)
query, params = gen.merge_relationship(
    {
        "__label__": "DIRECTED",
        "__source_uid__": "Lana Wachowski",
        "__target_uid__": "The Matrix",
    }
)
print("MERGE relationship:")
print("  Query: ", query)
print("  Params:", params)

CREATE relationship:
  Query:  MATCH (a:Person {name: $src_uid}), (b:Movie {title: $tgt_uid}) CREATE (a)-[r:ACTED_IN {role: $role}]->(b) RETURN r
  Params: {'src_uid': 'Keanu Reeves', 'tgt_uid': 'The Matrix', 'role': 'Neo'}

MERGE relationship:
  Query:  MATCH (a:Person {name: $src_uid}), (b:Movie {title: $tgt_uid}) MERGE (a)-[r:DIRECTED]->(b) RETURN r
  Params: {'src_uid': 'Lana Wachowski', 'tgt_uid': 'The Matrix'}


## Generating constraints

The generator can produce uniqueness constraint statements from the `uid_field` definitions in the model. These are typically run once when setting up the database schema.

In [7]:
constraints = gen.generate_constraints()
print(f"Generated {len(constraints)} constraint(s):\n")
for c in constraints:
    print(c)

Generated 3 constraint(s):

CREATE CONSTRAINT constraint_person_name IF NOT EXISTS FOR (n:Person) REQUIRE n.name IS UNIQUE
CREATE CONSTRAINT constraint_movie_title IF NOT EXISTS FOR (n:Movie) REQUIRE n.title IS UNIQUE
CREATE CONSTRAINT constraint_city_name IF NOT EXISTS FOR (n:City) REQUIRE n.name IS UNIQUE


## MATCH patterns

Generate read queries to retrieve nodes of a given type, or to match a specific relationship pattern.

In [8]:
# Match all Person nodes
print("Match Person:", gen.match_node(Person))
print()

# Match ACTED_IN relationship pattern
print("Match ACTED_IN:", gen.match_relationship(ActedIn))
print()

# Match LIVES_IN relationship pattern
print("Match LIVES_IN:", gen.match_relationship(LivesIn))

Match Person: MATCH (n:Person) RETURN n

Match ACTED_IN: MATCH (a:Person)-[r:ACTED_IN]->(b:Movie) RETURN a, r, b

Match LIVES_IN: MATCH (a:Person)-[r:LIVES_IN]->(b:City) RETURN a, r, b


## Putting it together

A typical database setup workflow: generate constraints first, then merge nodes, then create relationships. The output below shows the full sequence of Cypher statements that would populate a small filmography database.

In [9]:
print("=" * 60)
print("STEP 1: Create uniqueness constraints")
print("=" * 60)
for c in gen.generate_constraints():
    print(c)
print()

print("=" * 60)
print("STEP 2: Merge nodes")
print("=" * 60)
people = [
    {"__label__": "Person", "name": "Keanu Reeves", "born": 1964},
    {"__label__": "Person", "name": "Lana Wachowski", "born": 1965},
    {"__label__": "Person", "name": "Carrie-Anne Moss", "born": 1967},
]
movies = [
    {"__label__": "Movie", "title": "The Matrix", "year": 1999},
    {"__label__": "Movie", "title": "The Matrix Reloaded", "year": 2003},
]
cities = [
    {"__label__": "City", "name": "Los Angeles"},
]

for node_data in people + movies + cities:
    q, p = gen.merge_node(node_data)
    print(f"  {q}")
    print(f"    params: {p}")
print()

print("=" * 60)
print("STEP 3: Create relationships")
print("=" * 60)
relationships = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Carrie-Anne Moss",
        "__target_uid__": "The Matrix",
        "role": "Trinity",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix Reloaded",
        "role": "Neo",
    },
    {
        "__label__": "DIRECTED",
        "__source_uid__": "Lana Wachowski",
        "__target_uid__": "The Matrix",
    },
    {
        "__label__": "DIRECTED",
        "__source_uid__": "Lana Wachowski",
        "__target_uid__": "The Matrix Reloaded",
    },
    {
        "__label__": "LIVES_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "Los Angeles",
    },
]

for rel_data in relationships:
    q, p = gen.create_relationship(rel_data)
    print(f"  {q}")
    print(f"    params: {p}")

STEP 1: Create uniqueness constraints
CREATE CONSTRAINT constraint_person_name IF NOT EXISTS FOR (n:Person) REQUIRE n.name IS UNIQUE
CREATE CONSTRAINT constraint_movie_title IF NOT EXISTS FOR (n:Movie) REQUIRE n.title IS UNIQUE
CREATE CONSTRAINT constraint_city_name IF NOT EXISTS FOR (n:City) REQUIRE n.name IS UNIQUE

STEP 2: Merge nodes
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n
    params: {'name': 'Keanu Reeves', 'born': 1964}
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n
    params: {'name': 'Lana Wachowski', 'born': 1965}
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n
    params: {'name': 'Carrie-Anne Moss', 'born': 1967}
  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n
    params: {'title': 'The Matrix', 'year': 1999}
  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n
    params: {'title': 'The Matrix Reloaded', 'year': 2003}
  MERGE (n:City {name: $name}) RETURN n
    params: {'name': 'Los Angeles'}

STEP 3: 

## Undirected Relationship Queries

When a relationship type has `__directed__ = False`, the generated Cypher uses `-`
instead of `->` for both MATCH and CREATE/MERGE patterns. This tells the query engine
that direction does not matter.

This works for both same-type endpoints (e.g. `Person`-`Person`) and cross-type endpoints
(e.g. `Person`-`Company`).

In [10]:
# Add undirected relationships to the model
class Company(NodeModel):
    __label__ = "Company"
    __uid_field__ = "name"
    name: str


class FriendOf(RelationshipModel):
    __label__ = "FRIEND_OF"
    __source_label__ = "Person"
    __target_label__ = "Person"
    __directed__ = False
    since: Optional[int] = None


class Collaborates(RelationshipModel):
    __label__ = "COLLABORATES"
    __source_label__ = "Person"
    __target_label__ = "Company"
    __directed__ = False


extended_model = GraphDefinition(
    name="Extended",
    node_types=[Person, Movie, City, Company],
    relationship_types=[ActedIn, Directed, LivesIn, FriendOf, Collaborates],
)
gen2 = CypherGenerator(extended_model)
print("Extended model created with undirected relationships.")

Extended model created with undirected relationships.


In [11]:
# MATCH pattern for undirected relationship -- uses '-' instead of '->'
print("Directed MATCH:")
print(" ", gen2.match_relationship(ActedIn))
print()
print("Undirected MATCH (same-type):")
print(" ", gen2.match_relationship(FriendOf))
print()
print("Undirected MATCH (cross-type):")
print(" ", gen2.match_relationship(Collaborates))

Directed MATCH:
  MATCH (a:Person)-[r:ACTED_IN]->(b:Movie) RETURN a, r, b

Undirected MATCH (same-type):
  MATCH (a:Person)-[r:FRIEND_OF]-(b:Person) RETURN a, r, b

Undirected MATCH (cross-type):
  MATCH (a:Person)-[r:COLLABORATES]-(b:Company) RETURN a, r, b


In [12]:
# CREATE/MERGE for undirected relationships also use '-' instead of '->'
print("CREATE undirected relationship (with property):")
q, p = gen2.create_relationship(
    {
        "__label__": "FRIEND_OF",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "Carrie-Anne Moss",
        "since": 1999,
    }
)
print(f"  {q}")
print(f"  params: {p}")
print()

print("MERGE undirected cross-type relationship:")
q, p = gen2.merge_relationship(
    {
        "__label__": "COLLABORATES",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "Warner Bros",
    }
)
print(f"  {q}")
print(f"  params: {p}")
print()

# Compare with directed CREATE
print("CREATE directed relationship (for comparison):")
q, p = gen2.create_relationship(
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    }
)
print(f"  {q}")
print(f"  params: {p}")

CREATE undirected relationship (with property):
  MATCH (a:Person {name: $src_uid}), (b:Person {name: $tgt_uid}) CREATE (a)-[r:FRIEND_OF {since: $since}]->(b) RETURN r
  params: {'src_uid': 'Keanu Reeves', 'tgt_uid': 'Carrie-Anne Moss', 'since': 1999}

MERGE undirected cross-type relationship:
  MATCH (a:Person {name: $src_uid}), (b:Company {name: $tgt_uid}) MERGE (a)-[r:COLLABORATES]->(b) RETURN r
  params: {'src_uid': 'Keanu Reeves', 'tgt_uid': 'Warner Bros'}

CREATE directed relationship (for comparison):
  MATCH (a:Person {name: $src_uid}), (b:Movie {title: $tgt_uid}) CREATE (a)-[r:ACTED_IN {role: $role}]->(b) RETURN r
  params: {'src_uid': 'Keanu Reeves', 'tgt_uid': 'The Matrix', 'role': 'Neo'}


## Validating generated queries against the model

`CypherGenerator` ensures identifiers are safe and properties are model-declared at generation time.
But you can also run the same model-level validation that applies to any hand-written Cypher query:
`validate_cypher` checks a query string against the full `GraphDefinition` â€” labels, relationship
types, property names, and endpoint constraints.

This means generated queries are **first-class citizens** of the validation surface, not a special
case. Any query string, whether hand-written or generated, goes through the same checks.

In [13]:
from orthograph.cypher.parser import validate_cypher


# A generated query passes model validation
cypher, _ = gen.merge_node(
    {"__label__": "Person", "name": "Keanu Reeves", "born": 1964}
)
result = validate_cypher(cypher, graph_definition)
print(f"merge_node:         valid={result.is_valid} | {cypher}")

cypher = gen.match_relationship(ActedIn)
result = validate_cypher(cypher, graph_definition)
print(f"match_relationship: valid={result.is_valid} | {cypher}")
print()

# A hand-written query referencing a property outside the model fails the same check
cypher_bad = "MATCH (p:Person) RETURN p.salary"
result_bad = validate_cypher(cypher_bad, graph_definition)
print(f"hand-written bad:   valid={result_bad.is_valid} | {cypher_bad}")
for issue in result_bad.errors:
    print(f"  {issue}")

merge_node:         valid=True | MERGE (n:Person {name: $name}) SET n.born = $born RETURN n
match_relationship: valid=True | MATCH (a:Person)-[r:ACTED_IN]->(b:Movie) RETURN a, r, b

hand-written bad:   valid=False | MATCH (p:Person) RETURN p.salary
  [ERROR] QUERY_UNKNOWN_PROPERTY: Query accesses property 'salary' on Person which is not in the model (entity=Person.salary)


## Typed-query emission

The raw-string methods are useful when you already have a data dict at hand (e.g. a loading pipeline
or a one-off script). For application code, `CypherGenerator` can also emit **typed query objects**
â€” the same `CypherReadQuery` / `CypherWriteQuery` instances you would write by hand.

These are useful when you want:
- A **reusable, named query** that lives in a `QueryCatalogue` and can be introspected.
- A typed `Output` model so the executor returns `Person` / `Movie` instances, not raw dicts.
- The definition-time `$param` â†” `Params` alignment guarantee â€” the generator synthesises
  the `Params` model from `get_property_specs()`, so the two are always in sync.
- **Auto-generated CRUD** for a new node type without writing any Cypher by hand.

The label is baked in as a validated literal at synthesis time, so each query object is
model-specific (one per node type) but otherwise identical in shape to a hand-authored query.

In [14]:
# Generate typed query objects for Person
match_q = gen.match_by_uid_query(Person)
merge_q = gen.merge_query(Person)
create_q = gen.create_query(Movie)
delete_q = gen.delete_by_uid_query(Person)

print("Typed queries:")
print(
    f"  {match_q.name:30} backend={match_q.backend.value}  Output={match_q.Output.__name__}"
)
print(f"  {merge_q.name:30} backend={merge_q.backend.value}")
print(f"  {create_q.name:30} backend={create_q.backend.value}")
print(f"  {delete_q.name:30} backend={delete_q.backend.value}")
print()

# build() is pure -- no session needed
cypher, params = match_q.build(match_q.Params(name="Keanu Reeves"))
print("match_by_uid build:")
print(f"  cypher: {cypher}")
print(f"  params: {params}")
print()

cypher, params = merge_q.build(merge_q.Params(name="Keanu Reeves", born=1964))
print("merge build:")
print(f"  cypher: {cypher}")
print(f"  params: {params}")

Typed queries:
  match_person_by_uid            backend=cypher  Output=Person
  merge_person                   backend=cypher
  create_movie                   backend=cypher
  delete_person_by_uid           backend=cypher

match_by_uid build:
  cypher: MATCH (n:Person {name: $name}) RETURN n
  params: {'name': 'Keanu Reeves'}

merge build:
  cypher: MERGE (n:Person {name: $name}) SET n.born = $born RETURN n
  params: {'born': 1964, 'name': 'Keanu Reeves'}


## Registering generated queries in a catalogue

Because they are standard `CypherReadQuery` / `CypherWriteQuery` instances, generated typed queries
register in a `QueryCatalogue` and are validated by `validate_catalogue` exactly like hand-written
ones. There is no special path for generated queries.

In [15]:
from orthograph.cypher.validation import validate_query_catalogue
from orthograph.query.catalogue import QueryCatalogue


query_catalogue = QueryCatalogue()
query_catalogue.register_read(gen.match_by_uid_query(Person))
query_catalogue.register_read(gen.match_by_uid_query(Movie))
query_catalogue.register_write(gen.merge_query(Person))
query_catalogue.register_write(gen.create_query(Movie))
query_catalogue.register_write(gen.delete_by_uid_query(Person))

print("Registered queries:")
for desc in query_catalogue.describe():
    schema = " output_schema=Person" if desc.output_schema else ""
    print(f"  {desc.name:30} kind={desc.kind:5} backend={desc.backend.value}{schema}")
print()

result = validate_query_catalogue(query_catalogue, graph_definition)
print(f"validate_query_catalogue: valid={result.is_valid}")
for issue in result.issues:
    print(f"  {issue}")

Registered queries:
  match_person_by_uid            kind=read  backend=cypher output_schema=Person
  match_movie_by_uid             kind=read  backend=cypher output_schema=Person
  merge_person                   kind=write backend=cypher
  create_movie                   kind=write backend=cypher
  delete_person_by_uid           kind=write backend=cypher

validate_query_catalogue: valid=True
